In [1]:
import torch
from models.eegnet.model import EEGNet

# Initialize EEGNet with its accepted parameters
model = EEGNet(
    n_classes=2,               # Number of classes
    electrode_channels=64,     # Number of EEG channels
    sample_length=128,         # Number of time samples
    dropout_rate=0.5,          # Dropout probability
    kernel_length=32,          # Temporal convolution kernel length
    f1=8,                      # Number of temporal filters
    d=2,                       # Number of spatial filters per temporal filter
    f2=16                      # Pointwise convolution filters
)

print(model)

EEGNet(
  (block1_conv1): Conv2d(1, 8, kernel_size=(1, 32), stride=(1, 1), padding=same, bias=False)
  (block1_bn1): BatchNorm2d(8, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_conv2): Conv2d(8, 16, kernel_size=(64, 1), stride=(1, 1), groups=8, bias=False)
  (block1_bn2): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block1_elu): ELU(alpha=1.0)
  (block1_pool): AvgPool2d(kernel_size=(1, 4), stride=(1, 4), padding=0)
  (block1_dropout): Dropout(p=0.5, inplace=False)
  (block2_conv1): Conv2d(16, 16, kernel_size=(1, 16), stride=(1, 1), padding=same, groups=16, bias=False)
  (block2_conv2): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
  (block2_bn): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
  (block2_elu): ELU(alpha=1.0)
  (block2_pool): AvgPool2d(kernel_size=(1, 8), stride=(1, 8), padding=0)
  (block2_dropout): Dropout(p=0.5, inplace=False)
  (flatten): Flatten(start

# EEGNet Baseline Pipeline
This notebook establishes the formal baseline for Motor Imagery classification using EEGNet. 
It uses our structured workflow to log the dataset details and hyperparameters to TensorBoard.

In [2]:
from functools import partial

import torch
import torch.nn as nn
import torch.optim as optim
from moabb.datasets import BNCI2014_001, Gao2026
import datetime

# Import custom tools
from tools.dataloading import get_motorimagery_loaders
from tools.preprocessing import apply_mne_ica
from tools.training import Trainer
from models.eegnet.model import EEGNet

In [3]:
# 1. Pipeline Configuration
dataset = BNCI2014_001()

# Setting up standard industry parameters for testing a baseline
hyperparams = {
    "dataset": type(dataset).__name__,
    "split_mode": "loso",
    "test_subject_id": 1,
    "batch_size": 32,
    "sample_frequency": 250,
    "learning_rate": 0.001,
    "epochs": 300,
    "patience": 50,
    "dropout_rate": 0.5,
    "kernel_length": 64,
    "f1": 8,
    "d": 2,
    "f2": 16,
    "preprocessing": "ICA"
}

# Add dynamic timestamp string to organize logs
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
log_dir = f"models/eegnet/results/baseline_{hyperparams['dataset']}_{hyperparams['split_mode']}_{timestamp}"

In [4]:
custom_ica = partial(apply_mne_ica, n_components=15)

# 2. Data Loading
print(
    f"Loading {hyperparams['dataset']} with {hyperparams['split_mode'].upper()} split for Subject {hyperparams['test_subject_id']}...")

train_loader, test_loader, metadata = get_motorimagery_loaders(
    dataset,
    preprocessing_fn=custom_ica,
    sample_frequency=hyperparams["sample_frequency"],
    test_subject_id=hyperparams["test_subject_id"],
    batch_size=hyperparams["batch_size"],
    split_mode=hyperparams["split_mode"]
)

# Extract dynamic dimensions from the dataset that the model needs
hyperparams["n_classes"] = metadata["n_classes"]
hyperparams["electrode_channels"] = metadata["electrode_channels"]
hyperparams["sample_length"] = metadata["sample_length"]

print(
    f"Data ready. Found {hyperparams['n_classes']} classes across {hyperparams['electrode_channels']} channels.")

Choosing from all possible events


Loading BNCI2014_001 with LOSO split for Subject 1...
Applying MNE ICA (n_components=15) to epoched data...
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 15 components
Fitting ICA took 70.4s.
Applying ICA to Epochs instance
    Transforming to ICA space (15 components)
    Zeroing out 0 ICA components
    Projecting back using 22 PCA components


Split Mode: LOSO | Test Subject: 1
Training on 4608 trials from 8 subjects
Testing on 576 trials from 1 subject
Input Shape for Model: torch.Size([32, 1, 22, 1001])
Batch size: 32 | Nº Channels: 22 | Sample length: 1001
Data ready. Found 4 classes across 22 channels.


In [5]:
# 3. Model Initialization
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = EEGNet(
    n_classes=hyperparams["n_classes"],
    electrode_channels=hyperparams["electrode_channels"],
    sample_length=hyperparams["sample_length"],
    kernel_length=hyperparams["kernel_length"],
    dropout_rate=hyperparams["dropout_rate"],
    f1=hyperparams["f1"],
    d=hyperparams["d"],
    f2=hyperparams["f2"]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=hyperparams["learning_rate"])

Using device: cuda


In [6]:
# 4. Training and Evaluation Tracking
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    log_dir=log_dir,
    experiment_config=hyperparams
)

trainer.train(
    train_loader=train_loader,
    test_loader=test_loader,
    epochs=hyperparams["epochs"],
    patience=hyperparams["patience"]
)

Starting training on cuda...
Logging TensorBoard to: models/eegnet/results/baseline_BNCI2014_001_loso_20260329-202943


/home/carlos/.conda/envs/ml/lib/python3.13/site-packages/torch/nn/modules/conv.py:543: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv2d(


Epoch [1/300] | Train Loss: 1.3742 | Test Loss: 1.3683 | Test Acc: 27.26%
Epoch [10/300] | Train Loss: 1.1477 | Test Loss: 0.9967 | Test Acc: 61.46%
Epoch [20/300] | Train Loss: 1.0681 | Test Loss: 0.9391 | Test Acc: 62.33%
Epoch [30/300] | Train Loss: 1.0317 | Test Loss: 0.9546 | Test Acc: 62.33%
Epoch [40/300] | Train Loss: 0.9992 | Test Loss: 0.9206 | Test Acc: 62.85%
Epoch [50/300] | Train Loss: 0.9791 | Test Loss: 0.8820 | Test Acc: 65.10%
Epoch [60/300] | Train Loss: 0.9439 | Test Loss: 0.9085 | Test Acc: 64.58%
Epoch [70/300] | Train Loss: 0.9459 | Test Loss: 0.8893 | Test Acc: 65.80%
Epoch [80/300] | Train Loss: 0.9419 | Test Loss: 0.9025 | Test Acc: 64.41%
Epoch [90/300] | Train Loss: 0.9247 | Test Loss: 0.8484 | Test Acc: 68.92%
Epoch [100/300] | Train Loss: 0.9085 | Test Loss: 0.9402 | Test Acc: 61.98%
Epoch [110/300] | Train Loss: 0.9331 | Test Loss: 0.8864 | Test Acc: 65.97%
Epoch [120/300] | Train Loss: 0.9049 | Test Loss: 0.9230 | Test Acc: 64.58%
Epoch [130/300] | Train